In [ ]:
import mne
from mne.viz import plot_bem, plot_cov
from mne.minimum_norm import apply_inverse_raw, apply_inverse_epochs, make_inverse_operator, write_inverse_operator
import os
import numpy as np
import numpy.linalg as la
import scipy.io
import gc
from IPython.display import Image
#from eeg_file_operations import raw_reststate_data_finder_one_file

# Set backend to 'notebook' to work with vtk-osmesa build for off-screen rendering on cluster
mne.viz.set_3d_backend('notebook')

In [ ]:
# Parameters
ppt_id = 'default_ppt'
subject = 'default_subject'
head_circumference = 0.58 # in m
tms_target = 'default_target'
filename = 'default_name'

# Read in eeg sources savedir
savedir = 'default_savedir'
subjects_dir = 'default_subjectsdir'

# Read paths to stc, raw preprocessed EEG, preprocessed phantom EEG
raw_path = 'default_raw_path'

In [ ]:
# Set n_jobs for Joblib parallel jobs
n_jobs = 2
print(f'Running refCOV computation for {filename}.')

# Read paths to EEG data, source estimates dir, and Freesurfer subjects dir
bem_src_trans_savedir = os.path.join(savedir, f'{ppt_id}_{tms_target}')
figs_savepath = os.path.join(savedir, 'sources_figs')
os.makedirs(bem_src_trans_savedir, exist_ok=True)
os.makedirs(os.path.join(figs_savepath, 'bem'), exist_ok=True)
os.makedirs(os.path.join(figs_savepath, 'bem_src'), exist_ok=True)

'''
Load the appropriate standard EEG montage

Without digitized electrode positions, a standard montage can be used to estimate
the 'trans' matrix during coregistration
'''
# Define and plot 257-Channel Geodesic EEG Channel Layout, adjusting for measured head circumference
head_radius = head_circumference / (2 * np.pi)
print(f'Approximate radius of head, based on a spherical model: {head_radius} m')

# Load a standard montage and scale to approx head radius. This is assuming digitized electrode positions are not available.
# Change the template montage to match your recording system. See the MNE documentation for a list of available montage templates.
GSN_montage = mne.channels.make_standard_montage('GSN-HydroCel-257', head_size=head_radius)

# Rename reference channel in standard DigMontage to match EEG (if necessary -- throws an error with mismatch)
#GSN_montage.rename_channels({'Cz':'E257'})  # EEGLAB naming convention is E257
GSN_montage.rename_channels({'Cz':'VREF'})  # EGI naming convention is VREF

# Load participant resting state EEG and Bart resting state EEG recording and reset bads list if necessary
raw = mne.io.read_raw_egi(raw_path, preload=True)  # read in .set files with read_raw_eeglab()

# Set raw montage to Geodesic Hydrocel 257
raw.set_montage(GSN_montage)

# Apply average reference projector (apply_inverse_raw() requires data to be average referenced)
raw.set_eeg_reference('average', projection=True)
raw.apply_proj()

# Define paths to bem solution, src, and coregistration -trans.fif file 
bem_path = os.path.join(bem_src_trans_savedir, f'{ppt_id}_{tms_target}-bem-solution.h5')
src_path = os.path.join(bem_src_trans_savedir, f'{ppt_id}_{tms_target}-cortical-src.fif')
trans_path = os.path.join(bem_src_trans_savedir, f'{ppt_id}_{tms_target}-refCOV-trans.fif')

In [ ]:
bem_path

In [ ]:
# Run BEM, src, and coregistration computation

# Check if MRI-based files exist. If so, read these in and skip to coregistation
if os.path.exists(bem_path):
    print('A boundary element model solution for this participant already exists. Loading bem-solution.h5')
    bem = mne.read_bem_solution(bem_path)
else:
    '''
    Visualize BEM surfaces (cortical surface, inner skull, outer skull, skin) and create full BEM model
    '''
    # Can plot bem on cluster -- this outputs a matplotlib fig and doesn't require pyvista
    plot_bem_kwargs = dict(
        subject=subject,
        subjects_dir=subjects_dir,
        brain_surfaces='white',
        orientation='coronal',
        slices=[30, 60, 90, 120, 150, 180])
    
    fig = plot_bem(**plot_bem_kwargs)
    fig.savefig(os.path.join(figs_savepath, 'bem', f'{ppt_id}_{tms_target}_bem_plot.png'), dpi=400)
    
    # Create the full BEM model of estimated electrical propogation in the source space from the precomputed BEM surfaces
    conductivity = (0.3, 0.006, 0.3)  # for three layers
    model = mne.make_bem_model(subject=subject, ico=4, conductivity=conductivity, subjects_dir=subjects_dir)  # ico controls surface subsampling
    bem = mne.make_bem_solution(model, solver='mne')
    
    # Save bem solution in subject's eeg_sources directory
    mne.write_bem_solution(bem_path, bem, overwrite=True)

# Check if MRI-based files exist. If so, read these in and skip to noise covariance estimation
if os.path.exists(src_path):
    print('Source space information for this participant already exists. Loading cortical-src.fif')
    src = mne.read_source_spaces(src_path)
else:
    '''
    Compute the Source Space (src)
    
    Define the position and orientation of the candidate source locations confined to the cortical surface
    '''
    src = mne.setup_source_space(subject, spacing='oct6', add_dist=True, subjects_dir=subjects_dir, n_jobs=n_jobs)
    print(src)
    fig = plot_bem(src=src, **plot_bem_kwargs)
    fig.savefig(os.path.join(figs_savepath, 'bem_src', f'{ppt_id}_{tms_target}_bem_src_plot.png'), dpi=400)
    # Write cortical source space to disk
    mne.write_source_spaces(src_path, src, overwrite=True)

In [ ]:
'''
Estimate the transformation from EEG sensor space to MRI space (coregistration)

The -trans and fwd files may be a little different for the preprocessed EEG vs raw EEG, 
as preprocessing may reduce the data rank after bad channel removal and interpolation.

So, create a new -trans file for the refCOV even if one already exists.
'''

# Set up an initial coreg model using subject-fiducials.fif if available or estimated fiducials from fsaverage
coreg = mne.coreg.Coregistration(raw.info, subject, subjects_dir, fiducials='auto')

# Learn an affine mapping from EEG-head space to MRI-head space, using the fiducial points for reference.
# Fit coreg model with subject fiducials.
coreg.fit_fiducials(verbose=True)

# Further optimize and refine the coregistration solution with the Iterative Closest Point (ICP) algorithm
coreg.fit_icp(
    n_iterations=30,   # Set max iterations
    nasion_weight=0, # Relative weight for nasion 
    lpa_weight=0,    # Relative weight for left pre-auricular point (LPA)
    rpa_weight=0,    # Relative weight for right pre-auricular point (RPA)
    hsp_weight=20,    # Improve scalp fit by weighting head shape points (HSP)
    verbose='INFO'
)

dists = coreg.compute_dig_mri_distances() * 1e3  # in mm
print(
    f'Distance between MRI-defined head shape points and template EEG electrode positions (mean/min/max):\n{np.mean(dists):.2f} mm '
    f'/ {np.min(dists):.2f} mm / {np.max(dists):.2f} mm'
)

# Save transformation file subject-trans.fif
print(f'Saving trans matrix to: {trans_path}')
trans = coreg.trans
print(f'Trans matrix:{trans}')
mne.write_trans(trans_path, trans, overwrite=True)

'''
Compute the forward solution (fwd; leadfield matrix)

Using the cortical source space, the coregistration -trans matrix,
and the BEM solution, now compute the leadfield matrix
'''
fwd = mne.make_forward_solution(
raw.info,
trans=trans,
src=src,
bem=bem,
eeg=True,  # eeg=True includes EEG computations
mindist=5.0,  # minimum distance of sources from inner skull surface (in mm)
n_jobs=n_jobs,
verbose=True,
)
print(fwd)

'''
Explore fwd

Fwd data is accessible via string keys. Check if any vertices were removed during the forward computation.
Forward computation can remove vertices that are too close to (or outside) the inner skull surface.
'''
print(f"Src before: {src}")
print(f"Src after:  {fwd['src']}")

# Access leadfield data directly
# The leadfield size will be equal to nsensors x nsources x 3 (for a free orientation model)
leadfield = fwd['sol']['data']
print(f'Leadfield size : {leadfield.shape[0]} sensors x {leadfield.shape[1]} dipoles')

# Average reference the .mat leadfield before saving
n = leadfield.shape[0]

R = np.eye(n) - np.ones((n, n))/n

leadfield_avgref = R @ leadfield

# Save leadfield
leadfield_savepath = os.path.join(savedir, 'full_leadfield_avgref')
os.makedirs(leadfield_savepath, exist_ok=True)
scipy.io.savemat(os.path.join(leadfield_savepath, f'{filename}_avref-fwd.mat'), {"L": leadfield_avgref})

# Compute refCOV for GEDAI and save
refCOV = leadfield_avgref @ leadfield_avgref.T

# Save refCOV
refCOV_savepath = os.path.join(savedir, 'GEDAI_refCOV')
os.makedirs(refCOV_savepath, exist_ok=True)
print(f'Saving participant refCOV under {refCOV_savepath}/{filename}_avref-refCOV.mat')
scipy.io.savemat(os.path.join(refCOV_savepath, f'{filename}_avref-refCOV.mat'), {"refCOV": refCOV})

In [ ]:
trans['trans']

In [ ]:
fwd['sol']['data'].shape

In [ ]:
# Set plotting parameters for mne.viz.plot_alignment
plot_kwargs = dict(
    subject=subject,
    subjects_dir=subjects_dir,
    surfaces=dict(seghead=0.7, brain=0.4),  # use seghead for high resolution head surface (lh.seghead)
    bem=bem,
    src=src,
    dig=True,  # plot digitized points and fiducials
    eeg=['original', 'projected'],  # plot eeg sensors and their projection onto the scalp
    show_axes=True,
    coord_frame='auto', fig=None
)
# Configure the view of the 3D space for plotting with mne.viz.set_3d_view
view_kwargs = dict(azimuth=80, elevation=90, distance=0.6, focalpoint=(0.0, 0.0, 0.0))

# Plot using notebook backend
fig = mne.viz.plot_alignment(raw.info, trans=trans, **plot_kwargs)
mne.viz.set_3d_view(fig, **view_kwargs)

# Take a screenshot — use the notebook backend’s plotter
alignment_fig_path = os.path.join(figs_savepath, 'coreg', f'{ppt_id}_{tms_target}_coreg_alignment.png')
fig.plotter.screenshot(alignment_fig_path)
Image(filename=alignment_fig_path)

In [ ]:
# Average reference the .mat leadfield before saving
# leadfield = fwd["sol"]["data"]   # channels × sources
# n = leadfield.shape[0]

# R = np.eye(n) - np.ones((n, n))/n

# leadfield_avgref = R @ leadfield
# refCOV = leadfield_avgref @ leadfield_avgref.T
refCOV

In [ ]:
refCOV.shape

In [ ]:
# 1. Symmetry (should be True)
print(np.allclose(refCOV, refCOV.T))

# 2. Rank should be n_channels - 1 (one rank lost to avg ref)
rank = np.linalg.matrix_rank(refCOV)
print(f"Rank: {rank}, expected: {n-1}")  # should be 256 for 257ch

# 3. Positive semi-definite — eigenvalues should all be >= 0
eigvals = np.linalg.eigvalsh(refCOV)
print(f"Min eigenvalue: {eigvals.min():.3e}")  # should be ~0, not negative
print(f"N near-zero eigenvalues: {np.sum(np.abs(eigvals) < 1e-6)}")  # should be 1

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1. Raw covariance matrix
im1 = axes[0].imshow(refCOV, cmap='RdBu_r', aspect='auto',
                     vmin=-np.percentile(np.abs(refCOV), 99),
                     vmax= np.percentile(np.abs(refCOV), 99))
axes[0].set_title('refCOV (raw values)')
axes[0].set_xlabel('Channel')
axes[0].set_ylabel('Channel')
plt.colorbar(im1, ax=axes[0])

# 2. Normalized/correlation form — easier to assess structure
# (removes amplitude differences between channels)
d = np.sqrt(np.diag(refCOV))
refCOV_norm = refCOV / np.outer(d, d)
im2 = axes[1].imshow(refCOV_norm, cmap='RdBu_r', aspect='auto', vmin=-1, vmax=1)
axes[1].set_title('refCOV (normalized)')
axes[1].set_xlabel('Channel')
plt.colorbar(im2, ax=axes[1])

# 3. Eigenspectrum — reveals rank and energy distribution
eigvals = np.linalg.eigvalsh(refCOV)[::-1]  # descending order
axes[2].semilogy(eigvals, 'k-', linewidth=1)
axes[2].axhline(y=1e-6, color='r', linestyle='--', label='near-zero threshold')
axes[2].set_title('Eigenspectrum')
axes[2].set_xlabel('Component')
axes[2].set_ylabel('Eigenvalue (log scale)')
axes[2].legend()

plt.tight_layout()
plt.show()